In [ ]:
from obstore.store import MemoryStore

store = MemoryStore()
put_result = store.put("a/b.txt", b"hello obstore")
print(f"put -> {put_result}")
assert put_result["e_tag"] is not None

result = store.get("a/b.txt")
data = bytes(result.bytes())
assert data == b"hello obstore"
meta = result.meta
print(f"meta -> {meta}")
assert meta["path"] == "a/b.txt"
assert meta["size"] == len(data)
print("MemoryStore put/get round-trip OK")

In [ ]:
# LocalStore: filesystem-backed store rooted at a temp directory.
import tempfile
from pathlib import Path

from obstore.store import LocalStore, from_url

tmp = tempfile.TemporaryDirectory()
store = LocalStore(tmp.name)
store.put("data/f.bin", b"0123456789")

assert (Path(tmp.name) / "data" / "f.bin").read_bytes() == b"0123456789"
head = store.head("data/f.bin")
print(f"head -> {head}")
assert head["size"] == 10

chunk = bytes(store.get_range("data/f.bin", start=2, end=6))
assert chunk == b"2345"

url_store = from_url(f"file://{tmp.name}")
assert isinstance(url_store, LocalStore)
print("LocalStore put on-disk + head/get_range + from_url OK")
tmp.cleanup()

In [ ]:
# Listing: paged list stream (collect) + delimiter-grouped listing.
from obstore.store import MemoryStore

store = MemoryStore()
for path in ["data/a.csv", "data/b.csv", "logs/x.log"]:
    store.put(path, b"payload")

entries = store.list().collect()
paths = sorted(m["path"] for m in entries)
assert paths == ["data/a.csv", "data/b.csv", "logs/x.log"]
print(f"list -> {paths}")

grouped = store.list_with_delimiter()
prefixes = sorted(grouped["common_prefixes"])
assert prefixes == ["data", "logs"]
assert grouped["objects"] == []
print(f"list_with_delimiter -> prefixes {prefixes}")

under_data = store.list_with_delimiter(prefix="data")
assert sorted(m["path"] for m in under_data["objects"]) == ["data/a.csv", "data/b.csv"]
print("list APIs OK")

In [ ]:
# Async API: every operation has an *_async twin; the list stream doubles as
# an async iterator. Top-level await rides Jupyter's running event loop
# (never asyncio.run in a notebook).
import obstore
from obstore.store import MemoryStore

store = MemoryStore()
await obstore.put_async(store, "k1", b"async-value")

result = await store.get_async("k1")
data = bytes(await result.bytes_async())
assert data == b"async-value"

listed = []
async for batch in store.list():
    listed.extend(m["path"] for m in batch)
assert listed == ["k1"]
print(f"async round-trip + async list OK -> {listed}")

In [ ]:
# Buffered file-like access + the not-found error path.
import obstore
from obstore.store import MemoryStore

store = MemoryStore()
writer = obstore.open_writer(store, "big/blob.bin")
writer.write(b"A" * 4096 + b"Z")
writer.close()

reader = obstore.open_reader(store, "big/blob.bin")
assert reader.size == 4097
assert bytes(reader.read(4)) == b"AAAA"
reader.seek(4096)
assert bytes(reader.read()) == b"Z"
print("open_writer/open_reader OK")

# Missing keys surface as the builtin FileNotFoundError in obstore 0.11
# (obstore.exceptions.NotFoundError exists for backend-specific raises).
try:
    store.get("nope")
    raise AssertionError("expected FileNotFoundError")
except FileNotFoundError as e:
    print(f"missing key -> FileNotFoundError: {str(e)[:60]}")